# Base vs GRPO benchmark on a Hugging Face GPU Job

This notebook (`RUN_HF_BENCHMARK_JOB.ipynb`) launches a **paid HF Job** (same pattern as `run_hf_training_job.ipynb`) that:

1. Clones your public GitHub repo (**single branch**, shallow).
2. Installs the `learn_handwriting` package and inference deps.
3. Runs [`BENCHMARK_GRPO_VS_BASE.py`](BENCHMARK_GRPO_VS_BASE.py) with **`--local`**: loads **Qwen2.5-7B-Instruct** and your **Hub LoRA** with Transformers/PEFT, evaluates **6 fixed episodes** (2 letters × easy / medium / hard), and prints **per-tier and overall coverage / success** plus **Δ GRPO − base**.

**Requirements:** HF account with Jobs enabled, a GPU flavor (default **L40S**), and **`HF_TOKEN`** passed as a Job secret so the container can download gated base weights and the adapter from the Hub.

**Repo layout:** after `git clone … /tmp/lh`, the Job expects `learn_handwriting/` at the root of the GitHub repo (same as training), i.e. `/tmp/lh/learn_handwriting/BENCHMARK_GRPO_VS_BASE.py`.

In [1]:
%pip install -q -U "huggingface_hub>=0.28.0"

/Users/abhijeetmishra/PycharmProjects/hackathon/scaler_8_april/learn_handwriting/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import time

from huggingface_hub import fetch_job_logs, get_token, inspect_job, login, run_job

/Users/abhijeetmishra/PycharmProjects/hackathon/scaler_8_april/learn_handwriting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Login

Uses `HF_TOKEN` from the environment if set. Otherwise run the browser flow.

In [3]:
if not get_token():
    login(add_to_git_credential=False)
else:
    print("Already logged in (HF_TOKEN / cache).")

Already logged in (HF_TOKEN / cache).


## 2. Job configuration

Edit **`GITHUB_PUBLIC_CLONE_URL`** and **`GIT_CLONE_BRANCH`** so the clone contains `BENCHMARK_GRPO_VS_BASE.py` and `datagen_sft/compare_base_vs_finetuned.py` (for `LocalModelClient`).

Set **`GRPO_MODEL_NAME`** to the Hub adapter you want to compare (default: your GRPO repo).

In [4]:
GITHUB_PUBLIC_CLONE_URL = "https://github.com/radharamanaa/OpenEnv-Learn-handwriting.git"
GIT_CLONE_BRANCH = "stage2/long_planning_english"

HF_USERNAME = "abhijeetmishra101"
BASE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
GRPO_MODEL_NAME = f"{HF_USERNAME}/Qwen2.5-7B-Handwriting-GRPO"

JOB_IMAGE = "pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel"
JOB_FLAVOR = "l40sx1"
JOB_TIMEOUT = "4h"

## 3. Launch benchmark Job

Passes **`HF_TOKEN`** as a secret so the Job can pull models from Hugging Face.

Stdout will show the same tables as running locally: **per difficulty (2 episodes each)** and **overall Δ mean coverage**.

In [5]:
token = get_token()
if not token:
    raise RuntimeError("No HF token. Run the login cell or set HF_TOKEN.")
if "OWNER/REPO" in GITHUB_PUBLIC_CLONE_URL or GITHUB_PUBLIC_CLONE_URL.count("github.com") == 0:
    raise ValueError(
        "Set GITHUB_PUBLIC_CLONE_URL to your public repo (green Code button on GitHub)."
    )

print("Clone:", GITHUB_PUBLIC_CLONE_URL, "branch:", GIT_CLONE_BRANCH)
print("Base:", BASE_MODEL_NAME)
print("GRPO adapter:", GRPO_MODEL_NAME)

remote_script = f"""
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq && apt-get install -y -qq --no-install-recommends git ca-certificates
export PIP_DISABLE_PIP_VERSION_CHECK=1
pip install -q -U pip wheel setuptools
git clone --depth 1 --single-branch --branch "${{GIT_CLONE_BRANCH}}" "${{GITHUB_PUBLIC_CLONE_URL}}" /tmp/lh
REPO_ROOT=/tmp/lh/learn_handwriting
if [ ! -f "$REPO_ROOT/BENCHMARK_GRPO_VS_BASE.py" ]; then
  echo "ERROR: expected $REPO_ROOT/BENCHMARK_GRPO_VS_BASE.py. Put learn_handwriting/ at the root of the GitHub repo (see training job notebook)." >&2
  ls -la /tmp/lh >&2 || true
  exit 1
fi
cd "$REPO_ROOT"
export PIP_ROOT_USER_ACTION=ignore
pip install -q -e "."
pip install -q "torch" "transformers>=4.44.0" "peft" "accelerate" "openai" "pydantic>=2"
export PYTHONPATH="$REPO_ROOT"
export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True
python BENCHMARK_GRPO_VS_BASE.py --local --verbose \\
  --base_model "${{BASE_MODEL_NAME}}" \\
  --grpo_model "${{GRPO_MODEL_NAME}}" \\
  --json_out "$REPO_ROOT/grpo_hf_benchmark_results.json"
echo "--- saved: $REPO_ROOT/grpo_hf_benchmark_results.json ---"
echo "Benchmark job finished."
"""

_job_env = {
    "GITHUB_PUBLIC_CLONE_URL": GITHUB_PUBLIC_CLONE_URL,
    "GIT_CLONE_BRANCH": GIT_CLONE_BRANCH,
    "BASE_MODEL_NAME": BASE_MODEL_NAME,
    "GRPO_MODEL_NAME": GRPO_MODEL_NAME,
}

job = run_job(
    image=JOB_IMAGE,
    command=["bash", "-c", remote_script],
    flavor=JOB_FLAVOR,
    timeout=JOB_TIMEOUT,
    secrets={"HF_TOKEN": token},
    env=_job_env,
)

print("Job:", job)
if getattr(job, "url", None):
    print("URL:", job.url)

Clone: https://github.com/radharamanaa/OpenEnv-Learn-handwriting.git branch: stage2/long_planning_english
Base: Qwen/Qwen2.5-7B-Instruct
GRPO adapter: abhijeetmishra101/Qwen2.5-7B-Handwriting-GRPO
Job: JobInfo(id='69ede08ad2c8bd8662bcfbdf', created_at=datetime.datetime(2026, 4, 26, 9, 53, 14, 952000, tzinfo=datetime.timezone.utc), docker_image='pytorch/pytorch:2.6.0-cuda12.4-cudnn9-devel', space_id=None, command=['bash', '-c', '\nset -euo pipefail\nexport DEBIAN_FRONTEND=noninteractive\napt-get update -qq && apt-get install -y -qq --no-install-recommends git ca-certificates\nexport PIP_DISABLE_PIP_VERSION_CHECK=1\npip install -q -U pip wheel setuptools\ngit clone --depth 1 --single-branch --branch "${GIT_CLONE_BRANCH}" "${GITHUB_PUBLIC_CLONE_URL}" /tmp/lh\nREPO_ROOT=/tmp/lh/learn_handwriting\nif [ ! -f "$REPO_ROOT/BENCHMARK_GRPO_VS_BASE.py" ]; then\n  echo "ERROR: expected $REPO_ROOT/BENCHMARK_GRPO_VS_BASE.py. Put learn_handwriting/ at the root of the GitHub repo (see training job note

## 4. (Optional) Poll status and stream logs

Uncomment or re-run with `job_id` from the cell above. Look for **Per difficulty** and **Δ GRPO-base** in the log tail.

In [ ]:
job_id = getattr(job, "id", None)
if job_id:
    for _ in range(120):
        info = inspect_job(job_id=job_id)
        st = info.status
        print(time.strftime("%H:%M:%S"), st)
        stage = getattr(st, "stage", str(st))
        if stage in ("COMPLETED", "ERROR", "CANCELLED"):
            break
        time.sleep(15)
    for line in fetch_job_logs(job_id=job_id):
        print(line, end="")
else:
    print("No job id on result; open the Job URL in the browser for logs.")

## Router-only variant (no GPU inference)

If you prefer the **Hugging Face Inference / router** instead of loading 7B inside the Job, change the remote script to **omit** `--local`, install only `pip install -q -e "." openai pydantic`, set `export API_BASE_URL=https://router.huggingface.co/v1`, and run `python BENCHMARK_GRPO_VS_BASE.py --verbose ...`. Both model ids must be accepted by that endpoint (merged GRPO checkpoint or a supported adapter deployment).